## Step 1 — Set up the correct GitHub project folder

In [13]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/traffic-accident-risk-analytics-git")
INPUT_FILE = PROJECT_DIR / "data" / "processed" / "crash_cleaned.csv"
REPORTS_DIR = PROJECT_DIR / "reports"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

assert (PROJECT_DIR / ".git").exists(), "GitHub repository was not cloned correctly."

print("Project:", PROJECT_DIR)
print("Cleaned dataset exists:", INPUT_FILE.exists())

Project: /content/traffic-accident-risk-analytics-git
Cleaned dataset exists: True


## Step 2 — Install Git LFS and retrieve the cleaned dataset

In [14]:
print("Cleaned file exists:", INPUT_FILE.exists())

if INPUT_FILE.exists():
    print("File size (MB):", round(INPUT_FILE.stat().st_size / (1024 * 1024), 2))

Cleaned file exists: True
File size (MB): 234.79


## Step 3 — Load the dataset

In [15]:
import pandas as pd
import numpy as np
import json

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE, low_memory=False)

print("Dataset shape:", df.shape)
print("First five rows:")
display(df.head())

Dataset shape: (450491, 184)
First five rows:


,Crash_ID,Crash_Fatal_Fl,Cmv_Involv_Fl,Schl_Bus_Fl,Rr_Relat_Fl,Medical_Advisory_Fl,Amend_Supp_Fl,Active_School_Zone_Fl,Crash_Date,Crash_Time,...,Source_Year,Secondary_Crash_Fl,Rpt_Dir_Traffic,Rpt_Sec_Speed_Limit,Investigat_Time_Rwy_Clrd,Investigat_Time_Scn_Clrd,Investigat_Notify_Date,Investigat_Arrv_Date,Investigat_Date_Rwy_Clrd,Investigat_Date_Scn_Clrd
0,13808771,N,N,N,N,N,Y,N,2021-04-21,04:50 PM,...,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17013970,N,N,N,N,N,Y,N,2021-07-16,09:56 AM,...,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,17785987,N,N,N,N,N,Y,N,2021-07-25,08:46 AM,...,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,18038241,N,N,N,N,N,N,N,2021-01-01,03:48 AM,...,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,18038267,N,N,N,N,N,N,N,2021-01-01,03:19 AM,...,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 4 — Inspect the available feature columns

In [16]:
keywords = [
    "date", "time", "sev", "county", "city",
    "latitude", "longitude", "speed", "weather",
    "light", "road", "intersection", "construction",
    "injury", "death", "fatal"
]

for col in df.columns:
    if any(word in col.lower() for word in keywords):
        print(col)

Crash_Fatal_Fl
Crash_Date
Crash_Time
Rpt_City_ID
Rpt_Outside_City_Limit_Fl
Rpt_Latitude
Rpt_Longitude
Rpt_Road_Part_ID
Toll_Road_Fl
Crash_Speed_Limit
Road_Constr_Zone_Fl
Road_Constr_Zone_Wrkr_Fl
Rpt_Sec_Road_Part_ID
Light_Cond_ID
Entr_Road_ID
Road_Type_ID
Road_Algn_ID
Investigat_Notify_Time
Investigat_Arrv_Time
Report_Date
Road_Part_Adj_ID
Road_Cls_ID
Road_Relat_ID
City_ID
Latitude
Longitude
Crash_Sev_ID
Roadbed_Width
Roadway_Width
Death_Cnt
Rpt_Sec_Speed_Limit
Investigat_Time_Rwy_Clrd
Investigat_Time_Scn_Clrd
Investigat_Notify_Date
Investigat_Arrv_Date
Investigat_Date_Rwy_Clrd
Investigat_Date_Scn_Clrd


## Step 5 — Create time features

In [17]:
df["Crash_Date"] = pd.to_datetime(df["Crash_Date"], errors="coerce")

df["Crash_Year"] = df["Crash_Date"].dt.year
df["Crash_Month"] = df["Crash_Date"].dt.month
df["Crash_DayOfWeek"] = df["Crash_Date"].dt.dayofweek
df["Is_Weekend"] = df["Crash_DayOfWeek"].isin([5, 6]).astype("int8")

def extract_hour(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    # Handle HH:MM or HH:MM:SS
    parsed = pd.to_datetime(s, errors="coerce")
    if pd.notna(parsed) and ":" in s:
        return parsed.hour

    # Handle numeric HHMM values, such as 1345 or 945
    try:
        n = float(s)
        if not n.is_integer():
            return np.nan

        digits = str(int(n)).zfill(4)
        hour, minute = int(digits[:2]), int(digits[2:])

        if 0 <= hour <= 23 and 0 <= minute <= 59:
            return hour
    except (ValueError, TypeError):
        pass

    return np.nan

df["Crash_Hour"] = df["Crash_Time"].apply(extract_hour)

df["Time_Period"] = pd.cut(
    df["Crash_Hour"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["Night", "Morning", "Afternoon", "Evening", "Late_Night"]
)

display(df[[
    "Crash_Date", "Crash_Time", "Crash_Year",
    "Crash_Month", "Crash_DayOfWeek",
    "Is_Weekend", "Crash_Hour", "Time_Period"
]].head(10))

,Crash_Date,Crash_Time,Crash_Year,Crash_Month,Crash_DayOfWeek,Is_Weekend,Crash_Hour,Time_Period
0,2021-04-21,04:50 PM,2021,4,2,0,16,Afternoon
1,2021-07-16,09:56 AM,2021,7,4,0,9,Morning
2,2021-07-25,08:46 AM,2021,7,6,1,8,Morning
3,2021-01-01,03:48 AM,2021,1,4,0,3,Night
4,2021-01-01,03:19 AM,2021,1,4,0,3,Night
5,2021-01-01,02:16 AM,2021,1,4,0,2,Night
6,2021-01-01,01:30 AM,2021,1,4,0,1,Night
7,2021-01-01,07:05 AM,2021,1,4,0,7,Morning
8,2021-01-01,10:38 AM,2021,1,4,0,10,Morning
9,2021-01-01,01:50 AM,2021,1,4,0,1,Night


## Step 6 — Define the severity target

In [18]:
df["Crash_Sev_ID"] = pd.to_numeric(df["Crash_Sev_ID"], errors="coerce")

severity_labels = {
    1: "Serious_Injury",
    2: "Minor_Injury",
    3: "Possible_Injury",
    4: "Fatal_Injury",
    5: "Not_Injured"
}

df["Severity_Target"] = df["Crash_Sev_ID"].map(severity_labels)

display(
    df["Severity_Target"]
    .value_counts(dropna=False)
    .rename_axis("Severity_Target")
    .reset_index(name="Record_Count")
)

,Severity_Target,Record_Count
0,Not_Injured,273456
1,Possible_Injury,69741
2,Minor_Injury,62226
3,NaN,30181
4,Serious_Injury,12322
5,Fatal_Injury,2565


## Step 7 — Select predictors safely

In [19]:
context_candidates = [
    "County_ID",
    "City_ID",
    "Latitude",
    "Longitude",
    "Speed_Limit",
    "Weather_Cond_ID",
    "Light_Cond_ID",
    "Road_Cont_ID",
    "Intrsct_Relat_ID",
    "Construction_Zone_Fl"
]

context_features = [
    col for col in context_candidates
    if col in df.columns
]

time_features = [
    "Crash_Month",
    "Crash_DayOfWeek",
    "Is_Weekend",
    "Crash_Hour",
    "Time_Period"
]

predictor_columns = time_features + context_features

print("Time features:", time_features)
print("Context features found:", context_features)
print("Total candidate predictors:", len(predictor_columns))

Time features: ['Crash_Month', 'Crash_DayOfWeek', 'Is_Weekend', 'Crash_Hour', 'Time_Period']
Context features found: ['City_ID', 'Latitude', 'Longitude', 'Light_Cond_ID', 'Intrsct_Relat_ID']
Total candidate predictors: 10


## Step 8 — Create and save the engineered dataset

In [20]:
metadata_columns = [
    c for c in ["Crash_ID", "Source_Year", "Crash_Year"]
    if c in df.columns
]

modeling_data = df[
    metadata_columns + predictor_columns + ["Severity_Target"]
].copy()

output_file = PROCESSED_DIR / "feature_engineered.csv"
modeling_data.to_csv(output_file, index=False)

pd.DataFrame({"Feature": predictor_columns}).to_csv(
    REPORTS_DIR / "feature_engineering_feature_list.csv",
    index=False
)

print("Saved:", output_file)
print("Shape:", modeling_data.shape)
print("Rows retained:", len(modeling_data))

Saved: /content/traffic-accident-risk-analytics-git/data/processed/feature_engineered.csv
Shape: (450491, 14)
Rows retained: 450491


## Step 9 — Save validation reports

In [21]:
# Target distribution
df["Severity_Target"].value_counts(dropna=False).rename_axis(
    "Severity_Target"
).reset_index(name="Record_Count").to_csv(
    REPORTS_DIR / "feature_engineering_target_distribution.csv",
    index=False
)

# Missing values in predictors
pd.DataFrame({
    "Feature": predictor_columns,
    "Missing_Count": [modeling_data[c].isna().sum() for c in predictor_columns],
    "Missing_Percent": [
        round(modeling_data[c].isna().mean() * 100, 2)
        for c in predictor_columns
    ]
}).to_csv(
    REPORTS_DIR / "feature_engineering_missing_values.csv",
    index=False
)

print("Feature-engineering reports saved.")

Feature-engineering reports saved.


## Step 10 — Final validation

In [22]:
assert len(modeling_data) == 450491, "Unexpected row count."
assert "Crash_Sev_ID" not in predictor_columns
assert "Crash_ID" not in predictor_columns
assert output_file.exists()

print("PHASE 4 CHECK")
print("Dataset shape:", modeling_data.shape)
print("All original records retained:", len(modeling_data) == len(df))
print("Target leakage basic check: passed")
print("Output file size (MB):", round(output_file.stat().st_size / (1024 * 1024), 2))
print("Output:", output_file)

PHASE 4 CHECK
Dataset shape: (450491, 14)
All original records retained: True
Target leakage basic check: passed
Output file size (MB): 34.24
Output: /content/traffic-accident-risk-analytics-git/data/processed/feature_engineered.csv


## Step 11 — Add files to Github

In [58]:
%cd /content

!git clone https://github.com/rishitha-2602/traffic-accident-risk-analytics.git traffic-project

/content
Cloning into 'traffic-project'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 157 (delta 12), reused 45 (delta 5), pack-reused 99 (from 1)
Receiving objects: 100% (157/157), 118.10 MiB | 18.63 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Updating files: 100% (86/86), done.


In [61]:
from pathlib import Path

PROJECT_DIR = Path("/content/traffic-project")
output_file = PROJECT_DIR / "data/processed/feature_engineered.csv"

output_file.parent.mkdir(parents=True, exist_ok=True)

modeling_data.to_csv(output_file, index=False)

print("File saved:", output_file)
print("File exists:", output_file.exists())
print("File size (MB):", round(output_file.stat().st_size / (1024 * 1024), 2))

File saved: /content/traffic-project/data/processed/feature_engineered.csv
File exists: True
File size (MB): 34.24


In [65]:
!git config --global user.name "rishitha-2602"
!git config --global user.email "gaddamrishithareddy2603@gmail.com"

In [66]:
%cd /content/traffic-project
!git add -f data/processed/feature_engineered.csv
!git commit -m "Add feature engineered dataset"

/content/traffic-project
[main bd36342] Add feature engineered dataset
 1 file changed, 450492 insertions(+)
 create mode 100644 data/processed/feature_engineered.csv


In [70]:
import os
import subprocess
import tempfile
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN secret is missing"

askpass_script = """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) exit 1 ;;
esac
"""

with tempfile.NamedTemporaryFile(mode="w", delete=False) as f:
    f.write(askpass_script)
    askpass = f.name

os.chmod(askpass, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = askpass
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    result = subprocess.run(
        ["git", "push", "origin", "main"],
        cwd="/content/traffic-project",
        env=env,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print("Push successful!")
finally:
    os.remove(askpass)
    token = None


Push successful!
